# Animals DataSet

## Read & Merge datasets after Extraction


In [2]:
import pandas as pd


# Load the datasets
file1_path = r"C:\Users\Ezra\Downloads\web_imputation_20_01_2025 (2).csv"
file2_path = r"C:\Users\Ezra\Downloads\web_imputation_21_01_2025.csv"

df1 = pd.read_csv(file1_path)
df2 = pd.read_csv(file2_path)

# Display the column names and a sample of the data
df1_info = df1.head()
df2_info = df2.head()
df1_columns = df1.columns.tolist()
df2_columns = df2.columns.tolist()

(df1_info, df2_info, df1_columns, df2_columns)


(                                          animal endangered  \
 0   http://dbpedia.org/resource/ABC_Islands_bear         No   
 1        http://dbpedia.org/resource/Abida_ateni        Yes   
 2    http://dbpedia.org/resource/Abida_attenuata         No   
 3  http://dbpedia.org/resource/Abida_bigerrensis         No   
 4   http://dbpedia.org/resource/Abida_cylindrica         No   
 
   conservation status  average weight of the species  reproductive rate  \
 0           protected                          150.0                  2   
 1           protected                            5.0                  2   
 2           protected                            2.5                  3   
 3           protected                            2.5                  3   
 4           protected                            1.5                  3   
 
    primary predators  average lifespan in the wild              habitat range  
 0     tigers, humans                          15.0  forest, mountainous ar

# Data Processing

In [3]:
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer

# Normalize 'animal' field
def normalize_animal_column(df, column_name):
    return df[column_name].apply(lambda url: url.split("/")[-1] if isinstance(url, str) else url)

df1['animal_clean'] = normalize_animal_column(df1, 'animal')
df2['animal_clean'] = normalize_animal_column(df2, 'animal')

# Merge datasets
merged_df = pd.merge(df1, df2, on='animal_clean', suffixes=('_df1', '_df2'))

# Create combined_endangered
def compute_combined_endangered(row):
    val1 = 1 if row['endangered'].strip().lower() == 'yes' else 0
    val2 = 1 if row['endangeredStatus'].strip().lower() == 'yes' else 0
    return val1 + val2
merged_df['combined_endangered'] = merged_df.apply(compute_combined_endangered, axis=1)

# Copy reproductive_rate as-is
merged_df['reproductive_rate'] = merged_df['reproductive rate']

# --- Feature processing ---
def clean_and_split(text, delimiters=[',', ';']):
    if pd.isnull(text):
        return []
    for d in delimiters:
        text = text.replace(d, ';')
    return [item.strip().lower().rstrip('s') for item in text.split(';') if item.strip()]

# 1. One-hot encode conservation status
for col in ['conservation status_df1', 'conservation status_df2']:
    merged_df[col] = merged_df[col].str.strip().str.lower()
merged_df = pd.concat([
    merged_df,
    pd.get_dummies(merged_df['conservation status_df1'], prefix='status_df1'),
    pd.get_dummies(merged_df['conservation status_df2'], prefix='status_df2')
], axis=1)

# 2. Process predators
merged_df['primary_predators_clean'] = merged_df['primary predators'].apply(clean_and_split)
merged_df['predators_clean'] = merged_df['predators'].apply(clean_and_split)
merged_df['all_predators'] = merged_df.apply(
    lambda row: list(set(row['primary_predators_clean'] + row['predators_clean'])), axis=1)
mlb_predators = MultiLabelBinarizer()
predator_dummies = pd.DataFrame(mlb_predators.fit_transform(merged_df['all_predators']),
                                 columns=[f'predator_{p}' for p in mlb_predators.classes_])
merged_df = pd.concat([merged_df, predator_dummies], axis=1)

# 3. Process habitat range
merged_df['habitat_clean'] = merged_df['habitat range'].apply(clean_and_split)
mlb_habitats = MultiLabelBinarizer()
habitat_dummies = pd.DataFrame(mlb_habitats.fit_transform(merged_df['habitat_clean']),
                                columns=[f'habitat_{h}' for h in mlb_habitats.classes_])
merged_df = pd.concat([merged_df, habitat_dummies], axis=1)

# 4. Geographic distribution
def extract_region_and_coords(text):
    if pd.isnull(text) or ';' not in text:
        return pd.Series(['unknown', np.nan, np.nan])
    region, coord = text.split(';')[0], text.split(';')[1]
    try:
        lat, lon = map(float, coord.strip().split(','))
    except:
        lat, lon = np.nan, np.nan
    return pd.Series([region.strip().lower(), lat, lon])
merged_df[['geo_region', 'latitude', 'longitude']] = merged_df['geographic distribution'].apply(extract_region_and_coords)
merged_df['geo_region'] = merged_df['geo_region'].fillna('unknown')
geo_region_dummies = pd.get_dummies(merged_df['geo_region'], prefix='region')
merged_df = pd.concat([merged_df, geo_region_dummies], axis=1)

# 5. Breeding habits
def parse_breeding_habits(text):
    parts = clean_and_split(text, delimiters=[';', ','])
    return pd.Series({
        'breeds_in_spring': int('spring' in parts),
        'nests_on_ground': int(any('ground' in p for p in parts)),
        'offspring_range': int(any(any(char.isdigit() for char in p) for p in parts)),
    })
merged_df = pd.concat([merged_df, merged_df['breeding habits'].apply(parse_breeding_habits)], axis=1)

# 6. Human impact
merged_df['human_impact_clean'] = merged_df['human impact'].apply(clean_and_split)
mlb_impact = MultiLabelBinarizer()
impact_dummies = pd.DataFrame(mlb_impact.fit_transform(merged_df['human_impact_clean']),
                               columns=[f'human_impact_{i}' for i in mlb_impact.classes_])
merged_df = pd.concat([merged_df, impact_dummies], axis=1)

# 7. Climate change effects
merged_df['climate_effects_clean'] = merged_df['climate change effects'].apply(clean_and_split)
mlb_climate = MultiLabelBinarizer()
climate_dummies = pd.DataFrame(mlb_climate.fit_transform(merged_df['climate_effects_clean']),
                                columns=[f'climate_effect_{e}' for e in mlb_climate.classes_])
merged_df = pd.concat([merged_df, climate_dummies], axis=1)

# Save final processed dataset
merged_df.to_csv("processed_animal_endangerment_dataset.csv", index=False)

# Helper to clean and split multi-category fields
def clean_and_split(text, delimiters=[',', ';']):
    if pd.isnull(text):
        return []
    for d in delimiters:
        text = text.replace(d, ';')
    return [item.strip().lower().rstrip('s') for item in text.split(';') if item.strip()]

# --- 1. One-hot encode conservation status from both datasets ---
for col in ['conservation status_df1', 'conservation status_df2']:
    merged_df[col] = merged_df[col].str.strip().str.lower()

status_dummies_df1 = pd.get_dummies(merged_df['conservation status_df1'], prefix='status_df1')
status_dummies_df2 = pd.get_dummies(merged_df['conservation status_df2'], prefix='status_df2')
merged_df = pd.concat([merged_df, status_dummies_df1, status_dummies_df2], axis=1)

# --- 2. Multi-hot encode primary predators and predators ---
merged_df['primary_predators_clean'] = merged_df['primary predators'].apply(clean_and_split)
merged_df['predators_clean'] = merged_df['predators'].apply(clean_and_split)

# Combine and unify both predator lists
merged_df['all_predators'] = merged_df.apply(
    lambda row: list(set(row['primary_predators_clean'] + row['predators_clean'])), axis=1)

mlb_predators = MultiLabelBinarizer()
predator_dummies = pd.DataFrame(mlb_predators.fit_transform(merged_df['all_predators']),
                                 columns=[f'predator_{p}' for p in mlb_predators.classes_])
merged_df = pd.concat([merged_df, predator_dummies], axis=1)

# --- 3. Encode habitat range ---
merged_df['habitat_clean'] = merged_df['habitat range'].apply(clean_and_split)
mlb_habitats = MultiLabelBinarizer()
habitat_dummies = pd.DataFrame(mlb_habitats.fit_transform(merged_df['habitat_clean']),
                                columns=[f'habitat_{h}' for h in mlb_habitats.classes_])
merged_df = pd.concat([merged_df, habitat_dummies], axis=1)

# --- 4. Geographic distribution split ---
def extract_region_and_coords(text):
    if pd.isnull(text) or ';' not in text:
        return pd.Series(['unknown', np.nan, np.nan])
    region, coord = text.split(';')[0], text.split(';')[1]
    try:
        lat, lon = map(float, coord.strip().split(','))
    except:
        lat, lon = np.nan, np.nan
    return pd.Series([region.strip().lower(), lat, lon])

merged_df[['geo_region', 'latitude', 'longitude']] = merged_df['geographic distribution'].apply(extract_region_and_coords)

# One-hot encode region
merged_df['geo_region'] = merged_df['geo_region'].fillna('unknown')
geo_region_dummies = pd.get_dummies(merged_df['geo_region'], prefix='region')
merged_df = pd.concat([merged_df, geo_region_dummies], axis=1)

# --- 5. Breeding habits ---
def parse_breeding_habits(text):
    parts = clean_and_split(text, delimiters=[';', ','])
    features = {
        'breeds_in_spring': 0,
        'nests_on_ground': 0,
        'offspring_range': 0,
    }
    for part in parts:
        if 'spring' in part:
            features['breeds_in_spring'] = 1
        if 'ground' in part:
            features['nests_on_ground'] = 1
        if any(char.isdigit() for char in part):
            features['offspring_range'] = 1
    return pd.Series(features)

breeding_features = merged_df['breeding habits'].apply(parse_breeding_habits)
merged_df = pd.concat([merged_df, breeding_features], axis=1)

# --- 6. Human impact ---
merged_df['human_impact_clean'] = merged_df['human impact'].apply(clean_and_split)
mlb_impact = MultiLabelBinarizer()
impact_dummies = pd.DataFrame(mlb_impact.fit_transform(merged_df['human_impact_clean']),
                               columns=[f'human_impact_{i}' for i in mlb_impact.classes_])
merged_df = pd.concat([merged_df, impact_dummies], axis=1)

# --- 7. Climate change effects ---
merged_df['climate_effects_clean'] = merged_df['climate change effects'].apply(clean_and_split)
mlb_climate = MultiLabelBinarizer()
climate_dummies = pd.DataFrame(mlb_climate.fit_transform(merged_df['climate_effects_clean']),
                                columns=[f'climate_effect_{e}' for e in mlb_climate.classes_])
merged_df = pd.concat([merged_df, climate_dummies], axis=1)

# Display final processed dataset
#merged_df.to_csv("processed_animal_endangerment_dataset.csv", index=False)  # to save

df = merged_df

# Step 1: Create the label column BEFORE dropping anything
df['label'] = df['endangered'].str.strip().str.lower().map({'yes': 1, 'no': 0})

# Step 2: Drop all raw or unused columns
drop_cols = [
    'animal_df1', 'animal_df2', 'animal_clean',
    'primary predators', 'predators',
    'habitat range', 'breeding habits',
    'human impact', 'climate change effects',
    'geo_region', 'geographic distribution',
    'primary_predators_clean', 'predators_clean',
    'all_predators', 'habitat_clean',
    'human_impact_clean', 'climate_effects_clean',
    'conservation status_df1', 'conservation status_df2',
    'endangered',  # now safe to drop
]

df = df.drop(columns=drop_cols, errors='ignore')

# Step 3: Optionally drop combined and secondary label fields
df = df.drop(columns=['combined_endangered', 'endangeredStatus'], errors='ignore')

# Step 4: Save the cleaned file
updated_path = "./processed_animal_endangerment_dataset_with_true_labels.csv"
#df.to_csv(updated_path, index=False)


# Simple Evaluation of the Entire (fully imputed by the Extractor) Dataset

In [5]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

# Load your dataset
df = pd.read_csv(filepath_or_buffer=r'C:\Users\Ezra\PycharmProjects\evaluation of core sets selection methods\src\processed_animal_endangerment_dataset_with_true_labels.csv')
random_state =42
# Function to sanitize and deduplicate column names
def sanitize_column_names(columns):
    seen = {}
    clean_columns = []
    for col in columns:
        base = str(col).strip()
        base = base.replace(' ', '_').replace('.', '_').replace('/', '_').replace('[', '').replace(']', '').replace('<', '').replace('>', '')
        new_col = base
        i = 1
        while new_col in seen:
            new_col = f"{base}_{i}"
            i += 1
        seen[new_col] = True
        clean_columns.append(new_col)
    return clean_columns

# Function to prepare dataset
def prepare_dataset_for_xgboost(df, target_col='label'):
    X = df.drop(columns=[target_col])
    y = df[target_col]

    X.columns = sanitize_column_names(X.columns)

    # Replace NaNs and cast to float32
    X = X.fillna(-999).astype(np.float32)

    return X, y

# Function to prepare dataset
def prepare_df_for_xgboost(df, target_col='label'):

    df.columns = sanitize_column_names(df.columns)

    # Replace NaNs and cast to float32
    df = df.fillna(-999).astype(int)

    return df

X, y = prepare_dataset_for_xgboost(df)
df = prepare_df_for_xgboost(df)

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=random_state)

model = xgb.XGBClassifier(eval_metric='logloss')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("F1 Score:", f1_score(y_test, y_pred))

F1 Score: 0.43243243243243246


In [6]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

# Load your dataset
df = pd.read_csv(filepath_or_buffer=r'C:\Users\Ezra\PycharmProjects\evaluation of core sets selection methods\src\processed_animal_endangerment_dataset_with_true_labels.csv')

# Function to sanitize and deduplicate column names
def sanitize_column_names(columns):
    seen = {}
    clean_columns = []
    for col in columns:
        base = str(col).strip()
        base = base.replace(' ', '_').replace('.', '_').replace('/', '_').replace('[', '').replace(']', '').replace('<', '').replace('>', '')
        new_col = base
        i = 1
        while new_col in seen:
            new_col = f"{base}_{i}"
            i += 1
        seen[new_col] = True
        clean_columns.append(new_col)
    return clean_columns

# Function to prepare dataset
def prepare_dataset_for_xgboost(df, target_col='label'):
    X = df.drop(columns=[target_col])
    y = df[target_col]

    X.columns = sanitize_column_names(X.columns)

    # Replace NaNs and cast to float32
    X = X.fillna(-999).astype(np.float32)

    return X, y

# Function to prepare dataset
def prepare_df_for_xgboost(df, target_col='label'):

    df.columns = sanitize_column_names(df.columns)

    # Replace NaNs and cast to float32
    df = df.fillna(-999).astype(int)

    return df

X, y = prepare_dataset_for_xgboost(df)
df = prepare_df_for_xgboost(df)

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=random_state)

model = xgb.XGBClassifier(eval_metric='logloss')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("F1 Score:", f1_score(y_test, y_pred))

F1 Score: 0.43243243243243246



# Evaluate the Imputed DataSet, Icluding Prioritization
## Comparing 'ExtendTab' To 'Random Prioritization

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from src.dataset_with_indices_for_full_and_partial_data import Index_Dataset
from src.filtering_methods.filtering_methods_return_index import (RandomSampleFilter, ActiveLearningStartFromCoreSet, FilterEachIterXgboostPathSampleFinal)
import xgboost as xgb

random_state = 42

# ---------- 0. Helpers ----------
def sanitize_column_names(columns):
    seen = {}
    clean = []
    for col in columns:
        base = str(col).strip().replace(' ', '_').replace('.', '_') \
                               .replace('/', '_').replace('[', '') \
                               .replace(']', '').replace('<', '') \
                               .replace('>', '')
        new = base
        i = 1
        while new in seen:      # keep them unique
            new = f"{base}_{i}"
            i += 1
        seen[new] = True
        clean.append(new)
    return clean

# ---------- 1. Load & sanitize once ----------
target_col = "label"
df_original = pd.read_csv(
    r"C:\Users\Ezra\PycharmProjects\evaluation of core sets selection methods\src\processed_animal_endangerment_dataset_with_true_labels.csv"
)

feature_cols        = [c for c in df_original.columns if c != target_col]
df_original.columns = (
    sanitize_column_names(feature_cols) + [target_col]  # keep label untouched
)

# ---------- 2. Build X / y ----------
X = df_original.drop(columns=[target_col]).fillna(-999).astype(np.float32)
y = df_original[target_col]

# ---------- 3. Train / test split ----------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=random_state
)

# ---------- 4. Feed sanitized structures to Index_Dataset ----------
dataset_partial = Index_Dataset(
    df_original,      # contains the SAME sanitized names
    X_train,
    y_train,
    X_test,
    y_test,
    target_col,
    f1_score,
)

# ---------- 5. Run your filter ----------
all_results   = []
n = len(X_train)
core_set_sizes = [n//10, n//5, n//3, n // 2, int(0.75 * n), n]

for size in core_set_sizes:
    random_state += 1
    p          = size / n
    model      = xgb.XGBClassifier(eval_metric="logloss", random_state=random_state)
    sampler    = RandomSampleFilter(name="RandomSampleFilter",
                                    p=p, model=model)
    sampler.test_indices_filter_method(
        dataset_partial, dataset_partial, print_results=True
    )
    all_results.append(
        {"Method": "RandomSampleFilter",
         "Core Set Size": size,
         "F1 Score": sampler.scores,
         "Trial": sampler.run_times,}
    )

    initial_set_size = size //2
    # ExtendTAB Filter(Active Learning Start From Core Set)
    core_tab = FilterEachIterXgboostPathSampleFinal(
        'test_each_iter',
        trees_to_stop=30,
        threshold=5,
        examples_to_keep=size//2,
        prediction_model=model
    )
    core_set_sampler = core_tab.sample_indices(
        dataset_partial,
        p = initial_set_size / len(dataset_partial.X_train)
    )

    sampler = ActiveLearningStartFromCoreSet(name="ExtenTab",number_of_examples=size, start_size=size//2,core_set_sampler=core_set_sampler, batch_size=100,
                                     model=model)
    sampler.test_indices_filter_method(
        dataset_partial, dataset_partial, print_results=True
    )
    all_results.append(
        {"Method": "ExtenTab",
         "Core Set Size": size,
         "F1 Score": sampler.scores,
         "Trial": sampler.run_times}
    )


0 0.09974747474747475
Random Sampler: 0 236
FilteringResults(modeling_atitude='sample by partial dataset, train and test by the full dataset', score=0.36155606407322655, filtered_score=0.36155606407322655, run_time=0, new_size_percent=0.09932659932659933)
ExtendTab: len(initial_indices) 118 start_size 118
ExtendTab: AL Phase Current training set size: 118
ExtendTab: AL Phase Current training set size: 218
validate that the training set size is the wished size training set size: 318 wished core-set size: 237
FilteringResults(modeling_atitude='sample by partial dataset, train and test by the full dataset', score=0.35775862068965514, filtered_score=0.35775862068965514, run_time=0, new_size_percent=0.13383838383838384)
0 0.1999158249158249
Random Sampler: 0 470
FilteringResults(modeling_atitude='sample by partial dataset, train and test by the full dataset', score=0.36644591611479027, filtered_score=0.36644591611479027, run_time=0, new_size_percent=0.1978114478114478)
ExtendTab: len(initia

In [4]:

# Step 1: Flatten F1 scores into rows
flattened = []
for entry in all_results:
    method = entry['Method']
    core_size = entry['Core Set Size']
    for score in entry['F1 Score']:
        flattened.append({'Method': method, 'Core Set Size': core_size, 'F1 Score': score})

df = pd.DataFrame(flattened)
# Step 2: Group and summarize
summary = df.groupby(['Method', 'Core Set Size']).agg(
    avg_f1_score=('F1 Score', 'mean'),
    std_f1_score=('F1 Score', 'std'),
    num_trials=('F1 Score', 'count')
).reset_index()

# Optional: Mark best method per core set size
summary['is_best'] = summary.groupby('Core Set Size')['avg_f1_score'].transform(
    lambda x: x == x.max()
)

# Step 3: Print or export

In [5]:
(summary.sort_values(['Method', 'Core Set Size', 'avg_f1_score'], ascending=[True, True,False]))


,Method,Core Set Size,avg_f1_score,std_f1_score,num_trials,is_best
0,ExtenTab,237,0.357759,NaN,1,False
1,ExtenTab,475,0.458874,NaN,1,True
2,ExtenTab,792,0.453125,NaN,1,True
3,ExtenTab,1188,0.451883,NaN,1,True
4,ExtenTab,1782,0.463636,NaN,1,True
5,ExtenTab,2376,0.427518,NaN,1,True
6,RandomSampleFilter,237,0.361556,NaN,1,True
7,RandomSampleFilter,475,0.366446,NaN,1,False
8,RandomSampleFilter,792,0.396355,NaN,1,False
9,RandomSampleFilter,1188,0.425234,NaN,1,False


## ExtendTab Free Size

In [6]:
from src.filtering_methods.filtering_methods_return_index import (ActiveLearningStartFromCoreSet_based_on_reccommended_sampling_sizes)

# ===============================
#   ExtendTAB (Free size)
# ===============================
core_tab_hard_Samples = FilterEachIterXgboostPathSampleFinal(
    'test_each_iter',
    trees_to_stop=30,
    threshold=5,
    prediction_model=model,
    examples_to_keep=len(dataset_partial.X_train)//2,
)
core_set_sampler = core_tab_hard_Samples.sample_indices(
    dataset_partial,
    p=initial_set_size / len(dataset_partial.X_train)
)

filter_instance = ActiveLearningStartFromCoreSet_based_on_reccommended_sampling_sizes(
    name='ExtendTAB',
    start_size=len(dataset_partial.X_train)//2,
    number_of_examples=len(dataset_partial.X_train),
    model=model,
    core_set_sampler=core_set_sampler,
    batch_size=100
)

# Again, the critical fix: call the method
filter_instance.test_indices_filter_method(dataset_partial, dataset_partial, print_results=True)

all_results.append({
    #'Dataset': dataset_name,
    'Method': 'Extentab Free size',
    'Core Set Size': filter_instance.number_of_examples,
    'F1 Score': filter_instance.scores,
    'Trial': filter_instance.trials_number,
     'Data Used (%)': filter_instance.new_size_percents * 100
})


[ExtendTab] using supplied core set of 1188/1188
[ExtendTab] AL-phase — train size 1188
[ExtendTab] val-score 0.4062
[ExtendTab] AL-phase — train size 1288
[ExtendTab] val-score 0.4749
[ExtendTab] AL-phase — train size 1388
[ExtendTab] val-score 0.4656
[ExtendTab] AL-phase — train size 1488
[ExtendTab] val-score 0.4713
[ExtendTab] AL-phase — train size 1588
[ExtendTab] val-score 0.4484
[ExtendTab] AL-phase — train size 1688
[ExtendTab] val-score 0.4526
[ExtendTab] AL-phase — train size 1788
[ExtendTab] val-score 0.4809
[ExtendTab] AL-phase — train size 1888
[ExtendTab] val-score 0.4682
[ExtendTab] AL-phase — train size 1988
[ExtendTab] val-score 0.4736
[ExtendTab] AL-phase — train size 2088
[ExtendTab] val-score 0.4626
[ExtendTab] AL-phase — train size 2188
[ExtendTab] val-score 0.4203
[ExtendTab] AL-phase — train size 2288
[ExtendTab] val-score 0.4386
[ExtendTab] Stopped after 5 flat rounds.
[ExtendTab] initial 1188 → final 2376 (time 65.9s)
FilteringResults(modeling_atitude='sample b

In [9]:

# Step 1: Flatten F1 scores into rows
flattened = []
for entry in all_results:
    method = entry['Method']
    core_size = entry['Core Set Size']
    for score in entry['F1 Score']:
        flattened.append({'Method': method, 'Core Set Size': core_size, 'F1 Score': score})

df = pd.DataFrame(flattened)
# Step 2: Group and summarize
summary = df.groupby(['Method', 'Core Set Size']).agg(
    avg_f1_score=('F1 Score', 'mean'),
    std_f1_score=('F1 Score', 'std'),
    num_trials=('F1 Score', 'count')
).reset_index()

# Optional: Mark best method per core set size
summary['is_best'] = summary.groupby('Core Set Size')['avg_f1_score'].transform(
    lambda x: x == x.max()
)

# Step 3: Print or export
print(summary.sort_values(['Core Set Size', 'avg_f1_score'], ascending=[True, False]))

               Method  Core Set Size  avg_f1_score  std_f1_score  num_trials  \
0            ExtenTab            792      0.453846           0.0           3   
4  RandomSampleFilter            792      0.412698           0.0           3   
1            ExtenTab           1188      0.519685           0.0           3   
5  RandomSampleFilter           1188      0.413302           0.0           3   
2            ExtenTab           1782      0.465116           0.0           3   
6  RandomSampleFilter           1782      0.412322           0.0           3   
7  RandomSampleFilter           2376      0.439252           0.0           3   
3            ExtenTab           2376      0.430913           0.0           3   

   is_best  
0     True  
4    False  
1     True  
5    False  
2     True  
6    False  
7     True  
3    False  


In [10]:
summary

,Method,Core Set Size,avg_f1_score,std_f1_score,num_trials,is_best
0,ExtenTab,792,0.453846,0.0,3,True
1,ExtenTab,1188,0.519685,0.0,3,True
2,ExtenTab,1782,0.465116,0.0,3,True
3,ExtenTab,2376,0.430913,0.0,3,False
4,RandomSampleFilter,792,0.412698,0.0,3,False
5,RandomSampleFilter,1188,0.413302,0.0,3,False
6,RandomSampleFilter,1782,0.412322,0.0,3,False
7,RandomSampleFilter,2376,0.439252,0.0,3,True
